In [1]:
import pandas as pd
import numpy as np
import optuna
import json
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Load data
train = pd.read_csv('playground-series-s6e9/train.csv')
test = pd.read_csv('playground-series-s6e9/test.csv')
sample_sub = pd.read_csv('playground-series-s6e9/sample_submission.csv')

# Preprocessing
target = 'Will_Buy_EV'
features = [c for c in train.columns if c not in ['id', target]]

cat_cols = train[features].select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

train[target] = train[target].map({'Yes': 1, 'No': 0})

X = train[features]
y = train[target]
X_test = test[features]

num_trials = 50
n_splits = 5


In [2]:
import xgboost as xgb

def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'scale_pos_weight': float(np.sum(y == 0) / np.sum(y == 1)),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    auc_scores = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        
        model = xgb.train(params, dtrain, num_boost_round=1000, evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
        preds = model.predict(dval)
        auc = roc_auc_score(y_val, preds)
        auc_scores.append(auc)
        
    return np.mean(auc_scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=num_trials)
print("Best trial:", study.best_trial.params)

best_params = study.best_trial.params
best_params['objective'] = 'binary:logistic'
best_params['eval_metric'] = 'auc'
best_params['tree_method'] = 'hist'
best_params['scale_pos_weight'] = float(np.sum(y == 0) / np.sum(y == 1))

with open('xgboost_best_params.json', 'w') as f:
    json.dump(best_params, f)


[I 2026-09-10 19:33:32,338] A new study created in memory with name: no-name-19d98949-2046-4d49-9de1-a1a955d6a842


[I 2026-09-10 19:49:58,405] Trial 0 finished with value: 0.9418486092448134 and parameters: {'learning_rate': 0.04907562744740359, 'max_depth': 5, 'min_child_weight': 3, 'subsample': 0.9266197453216369, 'colsample_bytree': 0.8888194429659144, 'reg_alpha': 0.00010036489047706211, 'reg_lambda': 0.000847074926161031}. Best is trial 0 with value: 0.9418486092448134.


[I 2026-09-10 19:52:13,217] Trial 1 finished with value: 0.9370994620693684 and parameters: {'learning_rate': 0.001008524886297028, 'max_depth': 3, 'min_child_weight': 8, 'subsample': 0.9831304133913847, 'colsample_bytree': 0.5515062955642063, 'reg_alpha': 0.11390226728048161, 'reg_lambda': 3.3769373292054703e-07}. Best is trial 0 with value: 0.9418486092448134.


[I 2026-09-10 20:09:11,034] Trial 2 finished with value: 0.9419769916815561 and parameters: {'learning_rate': 0.07208956505800461, 'max_depth': 3, 'min_child_weight': 4, 'subsample': 0.812349263185093, 'colsample_bytree': 0.6386747772138543, 'reg_alpha': 9.744983129734327e-05, 'reg_lambda': 7.960882512856818e-08}. Best is trial 2 with value: 0.9419769916815561.


[I 2026-09-10 20:11:37,374] Trial 3 finished with value: 0.9386834932587774 and parameters: {'learning_rate': 0.0010512501837520148, 'max_depth': 5, 'min_child_weight': 1, 'subsample': 0.722311562653995, 'colsample_bytree': 0.7651172617966815, 'reg_alpha': 0.0008607229944879456, 'reg_lambda': 4.3754521423474175e-08}. Best is trial 2 with value: 0.9419769916815561.


...

[I 2026-09-11 08:52:49,293] Trial 46 finished with value: 0.9419133787882796 and parameters: {'learning_rate': 0.053570984500918156, 'max_depth': 3, 'min_child_weight': 5, 'subsample': 0.9466881131447948, 'colsample_bytree': 0.5844128481865339, 'reg_alpha': 0.2558332236496994, 'reg_lambda': 1.759195467045526e-08}. Best is trial 31 with value: 0.9420639904154646.


[I 2026-09-11 09:16:01,151] Trial 47 finished with value: 0.9419990042518306 and parameters: {'learning_rate': 0.04521523296784087, 'max_depth': 4, 'min_child_weight': 2, 'subsample': 0.904881579140623, 'colsample_bytree': 0.6138949529098655, 'reg_alpha': 0.15759917732485715, 'reg_lambda': 2.3252802588872776e-08}. Best is trial 31 with value: 0.9420639904154646.


[I 2026-09-11 09:35:26,979] Trial 48 finished with value: 0.9420247586359476 and parameters: {'learning_rate': 0.07904838290765964, 'max_depth': 3, 'min_child_weight': 3, 'subsample': 0.9057263778082623, 'colsample_bytree': 0.7222036503066747, 'reg_alpha': 0.033596865402509606, 'reg_lambda': 7.665680592558218e-08}. Best is trial 31 with value: 0.9420639904154646.


[I 2026-09-11 09:49:48,380] Trial 49 finished with value: 0.9419662189828022 and parameters: {'learning_rate': 0.0870624536996288, 'max_depth': 4, 'min_child_weight': 2, 'subsample': 0.9888331421339904, 'colsample_bytree': 0.6600411551894895, 'reg_alpha': 0.018035630244984873, 'reg_lambda': 5.647774284442727e-07}. Best is trial 31 with value: 0.9420639904154646.


Best trial: {'learning_rate': 0.081548291152698, 'max_depth': 3, 'min_child_weight': 5, 'subsample': 0.9584920067337729, 'colsample_bytree': 0.5279706274071247, 'reg_alpha': 0.21361143706392352, 'reg_lambda': 3.163023837161417e-08}


In [3]:
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
fold_preds_df = pd.DataFrame({'id': test['id']})

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    dtest = xgb.DMatrix(X_test)
    
    model = xgb.train(best_params, dtrain, num_boost_round=2000, evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
    
    val_preds = model.predict(dval)
    oof_preds[val_idx] = val_preds
    fold_test_preds = model.predict(dtest)
    test_preds += fold_test_preds / n_splits
    fold_preds_df[f'fold_{fold}'] = fold_test_preds

print("OOF AUC:", roc_auc_score(y, oof_preds))

pd.DataFrame({'id': train['id'], 'Will_Buy_EV': oof_preds}).to_csv('xgboost_oof_predictions.csv', index=False)
fold_preds_df.to_csv('xgboost_fold_predictions.csv', index=False)

sample_sub['Will_Buy_EV'] = test_preds
sample_sub.to_csv('submission_xgboost.csv', index=False)


OOF AUC: 0.942079784303414
